# Notebook 14 — TT-matrix の定義・Dense Reconstruction・TT-Linear Forward の理論と数値検証

## 今回の位置づけ

Notebook 13 では、TT tensor を dense tensor に復元せず、

$$
\langle\mathcal A,\mathcal B\rangle,
\qquad
\|\mathcal A\|_F,
\qquad
\|\mathcal A-\mathcal B\|_F
$$

を environment contraction だけで計算しました。

今回は NN に戻るための橋渡しとして、

$$
W\in\mathbb R^{m\times n}
$$

という dense `nn.Linear` の重みを **TT-matrix** として表現し、

$$
Y=XW^{\mathsf T}+b
$$

を dense weight $W$ を forward 中に復元せずに計算します。

## 今回の目的

1. TT-matrix core の定義を PyTorch tensor shape と対応させる
2. 行 index / 列 index の tensorization を理解する
3. grouped axis と interleaved axis の違いを理解する
4. TT-matrix cores から dense $W$ を復元する
5. dense $W$ を作らずに TT-Linear forward を実装する
6. TT-Linear の出力を、
   - `x @ W.T + bias`
   - `torch.nn.functional.linear`

   と比較する
7. rank-1 / 2-site の Kronecker 積ケースを確認する
8. dense weight を既存 TT-SVD へ渡すための tensorization を整理する

## 今回は扱わないもの

- backward
- optimizer
- 学習
- Fashion-MNIST
- CNN
- 大規模ベンチマーク
- TT-SVD 本体の再実装

## 重要

`tt_matrix_to_dense` は **小規模な数値検証専用**です。

本番の `tt_linear_forward` からは、dense weight を絶対に復元しません。


## 1. Dense Linear の重み規約

PyTorch の `nn.Linear(in_features=n, out_features=m)` は、

$$
W\in\mathbb R^{m\times n}
$$

という重みを持ちます。

batch 入力を、

$$
X\in\mathbb R^{B\times n},
$$

bias を、

$$
b\in\mathbb R^m
$$

とすると、

$$
\boxed{
Y=XW^{\mathsf T}+b
}
$$

です。

shape は、

$$
(B,n)(n,m)
\rightarrow
(B,m)
$$

です。

成分で書けば、

$$
Y_{b,i}
=
\sum_{j=0}^{n-1}
X_{b,j}W_{i,j}
+
b_i.
$$

つまり、

- 行 index $i$：出力 feature
- 列 index $j$：入力 feature

です。


## 2. 出力次元と入力次元を mode に分解する

TT-matrix を使うため、

$$
m=\prod_{k=1}^{d}m_k,
\qquad
n=\prod_{k=1}^{d}n_k
$$

と分解します。

ここで、

- $d$：TT site 数
- $m_k$：site $k$ の出力 mode size
- $n_k$：site $k$ の入力 mode size

です。

今回の主な数値例では、

$$
d=3,
$$

$$
(m_1,m_2,m_3)=(2,3,2),
$$

$$
(n_1,n_2,n_3)=(2,2,3)
$$

とします。

したがって、

$$
m=2\cdot3\cdot2=12,
$$

$$
n=2\cdot2\cdot3=12.
$$


## 3. 一本の index を multi-index にする

元の出力 index、

$$
i\in\{0,\ldots,m-1\}
$$

を、

$$
(i_1,\ldots,i_d)
$$

へ分解します。

PyTorch の通常の row-major / contiguous な並びでは、

$$
\boxed{
i
=
\sum_{k=1}^{d}
i_k
\prod_{\ell=k+1}^{d}m_\ell
}
$$

です。

同様に、

$$
\boxed{
j
=
\sum_{k=1}^{d}
j_k
\prod_{\ell=k+1}^{d}n_\ell
}
$$

です。

### $d=2$ の場合

$$
m=m_1m_2,
\qquad
n=n_1n_2
$$

なら、

$$
\boxed{
i=i_1m_2+i_2
}
$$

$$
\boxed{
j=j_1n_2+j_2
}
$$

です。

これは10進数で、

$$
42=4\cdot10+2
$$

と書くのと同じように、1本の index を「桁」に分けています。


## 4. TT-matrix の定義

通常の TT tensor core は、

$$
C^{(k)}
\in
\mathbb R^{r_{k-1}\times q_k\times r_k}
$$

でした。

TT-matrix では、site $k$ の物理 index を、

$$
a_k
\longleftrightarrow
(i_k,j_k)
$$

という **出力 mode / 入力 mode のペア**として扱います。

したがって第 $k$ core は、

$$
\boxed{
G^{(k)}
\in
\mathbb R^{
r_{k-1}
\times
m_k
\times
n_k
\times
r_k
}
}
$$

です。

axis 順序は必ず、

$$
\boxed{
(\text{left rank},\text{out mode},\text{in mode},\text{right rank})
}
$$

すなわち、

$$
(\alpha_{k-1},i_k,j_k,\alpha_k)
$$

に固定します。

境界 rank は、

$$
r_0=r_d=1.
$$


## 5. TT-matrix が表す dense weight

行 index を、

$$
i\leftrightarrow(i_1,\ldots,i_d),
$$

列 index を、

$$
j\leftrightarrow(j_1,\ldots,j_d)
$$

とします。

TT-matrix の要素は、

$$
\boxed{
W[i,j]
=
\sum_{\alpha_1,\ldots,\alpha_{d-1}}
\prod_{k=1}^{d}
G^{(k)}_{\alpha_{k-1},i_k,j_k,\alpha_k}
}
$$

です。

完全に書けば、

$$
\begin{aligned}
W[i,j]
=
\sum_{\alpha_1,\ldots,\alpha_{d-1}}
&
G^{(1)}_{0,i_1,j_1,\alpha_1}
G^{(2)}_{\alpha_1,i_2,j_2,\alpha_2}
\cdots
G^{(d)}_{\alpha_{d-1},i_d,j_d,0}.
\end{aligned}
$$

各 $(i_k,j_k)$ を固定すれば、

$$
G^{(k)}[:,i_k,j_k,:]
\in
\mathbb R^{r_{k-1}\times r_k}
$$

なので、通常の TT と同じように bond 行列の積になります。


## 6. $d=2$ の具体式

$d=2$ の場合、

$$
G^{(1)}
\in
\mathbb R^{1\times m_1\times n_1\times r},
$$

$$
G^{(2)}
\in
\mathbb R^{r\times m_2\times n_2\times1}.
$$

dense weight の要素は、

$$
\boxed{
W[i_1m_2+i_2,\;
  j_1n_2+j_2]
=
\sum_{\alpha=0}^{r-1}
G^{(1)}_{0,i_1,j_1,\alpha}
G^{(2)}_{\alpha,i_2,j_2,0}
}
$$

です。

この式が、

> TT-matrix core と通常の dense 行列 index の直接の対応

です。


## 7. なぜ axis permutation が必要か

dense weight、

$$
W\in\mathbb R^{m\times n}
$$

を mode 分解すると、まず自然に、

$$
\mathcal W_{\text{grouped}}
\in
\mathbb R^{
m_1\times\cdots\times m_d
\times
n_1\times\cdots\times n_d
}
$$

となります。

axis 順は、

$$
\boxed{
(i_1,i_2,\ldots,i_d,j_1,j_2,\ldots,j_d)
}
$$

です。

一方、TT-matrix の1 site は、

$$
(i_k,j_k)
$$

という組を担当します。

そのため TT-SVD に渡しやすい順番は、

$$
\boxed{
(i_1,j_1,i_2,j_2,\ldots,i_d,j_d)
}
$$

です。

したがって、

$$
(i_1,\ldots,i_d,j_1,\ldots,j_d)
\longrightarrow
(i_1,j_1,\ldots,i_d,j_d)
$$

という permutation が必要です。

一般の $d$ では permutation は、

$$
(0,d,1,d+1,\ldots,d-1,2d-1)
$$

です。

$d=3$ なら、

$$
\boxed{
[0,3,1,4,2,5]
}
$$

です。


## 8. Interleaved Tensor を通常 TT Tensor とみなす

axis permutation 後、

$$
\mathcal W_{\text{interleaved}}
\in
\mathbb R^{
m_1\times n_1\times
m_2\times n_2\times
\cdots\times
m_d\times n_d
}
$$

です。

site ごとに、

$$
q_k=m_kn_k
$$

とまとめれば、

$$
\boxed{
\mathcal A
\in
\mathbb R^{
q_1\times q_2\times\cdots\times q_d
}
}
$$

という通常の TT tensor にできます。

つまり、

$$
(i_k,j_k)
\longrightarrow
a_k=i_kn_k+j_k
$$

です。

この $\mathcal A$ に既存 TT-SVD を適用すれば、

$$
C^{(k)}
\in
\mathbb R^{r_{k-1}\times q_k\times r_k}
$$

が得られ、

$$
q_k=m_kn_k
$$

を戻して、

$$
\boxed{
G^{(k)}
=
C^{(k)}
.\operatorname{reshape}
(r_{k-1},m_k,n_k,r_k)
}
$$

とすれば TT-matrix core になります。


## 9. TT-matrix の parameter 数

dense weight の parameter 数は、

$$
\boxed{
P_{\text{dense}}
=
mn
}
$$

です。

TT-matrix は、

$$
\boxed{
P_{\text{TT}}
=
\sum_{k=1}^{d}
r_{k-1}m_kn_kr_k
}
$$

です。

TT-rank が小さく保てれば、大きな dense weight を少ない parameter で表せる可能性があります。

ただし、圧縮率は、

- mode 分解
- axis order
- weight の構造
- TT-rank

に強く依存します。


## 10. Dense Reconstruction の目的

TT-matrix cores を contract すると、まず、

$$
(i_1,j_1,i_2,j_2,\ldots,i_d,j_d)
$$

という interleaved tensor が得られます。

これを、

$$
(i_1,\ldots,i_d,j_1,\ldots,j_d)
$$

へ戻し、

$$
(m,n)
$$

へ reshape すれば dense $W$ になります。

今回 `tt_matrix_to_dense` は、

- TT-matrix 定義の確認
- permutation の確認
- TT-Linear forward の正解値生成

にだけ使います。

**本番 forward 内でこの関数を呼んではいけません。**


## 11. TT-Linear Forward の本質

入力は、

$$
X\in\mathbb R^{B\times n}
$$

です。

これを、

$$
\mathcal X
\in
\mathbb R^{
B\times n_1\times\cdots\times n_d
}
$$

へ reshape します。

TT-Linear forward は、

$$
\boxed{
(j_1,j_2,\ldots,j_d)
\longrightarrow
(i_1,i_2,\ldots,i_d)
}
$$

を core contraction だけで実現します。

各 site では、

$$
\boxed{
j_k\text{ を縮約して消し、}i_k\text{ を生成する}
}
$$

という操作を行います。

forward 中に、

$$
W\in\mathbb R^{m\times n}
$$

を作る必要はありません。


## 12. $d=3$ の Right-to-Left Forward

今回の実装では core を右から左へ縮約します。

入力は、

$$
X_{\text{tensor}}
[b,j_1,j_2,j_3]
$$

です。

### Step 3

第3 core、

$$
G^{(3)}[a_2,i_3,j_3,0]
$$

と $j_3$ を縮約します。

$$
\boxed{
T_3[b,j_1,j_2,i_3,a_2]
=
\sum_{j_3}
X_{\text{tensor}}[b,j_1,j_2,j_3]
G^{(3)}[a_2,i_3,j_3,0]
}
$$

shape は、

$$
(B,n_1,n_2,n_3)
\rightarrow
(B,n_1,n_2,m_3,r_2)
$$

です。

### Step 2

$$
\boxed{
T_2[b,j_1,i_2,i_3,a_1]
=
\sum_{j_2,a_2}
T_3[b,j_1,j_2,i_3,a_2]
G^{(2)}[a_1,i_2,j_2,a_2]
}
$$

shape は、

$$
(B,n_1,n_2,m_3,r_2)
\rightarrow
(B,n_1,m_2,m_3,r_1)
$$

です。

### Step 1

$$
\boxed{
Y_{\text{tensor}}[b,i_1,i_2,i_3]
=
\sum_{j_1,a_1}
T_2[b,j_1,i_2,i_3,a_1]
G^{(1)}[0,i_1,j_1,a_1]
}
$$

最終 shape は、

$$
(B,m_1,m_2,m_3)
$$

です。

最後に、

$$
(B,m)
$$

へ reshape し、bias を足します。


## 13. 実行環境と再現性設定


In [46]:
import math
import string

import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

device = torch.device("cpu")
dtype = torch.float64

print("PyTorch:", torch.__version__)
print("default dtype:", torch.get_default_dtype())
print("device:", device)


PyTorch: 2.11.0+cu128
default dtype: torch.float64
device: cpu


## 14. 既習レベルの補助関数

ここでは、

- Python sequence の積
- TT-matrix core の validation
- TT-rank の取得
- parameter 数の取得

を補助関数として用意します。

これらは今回の contraction アルゴリズム本体ではありません。


In [47]:
def product(values: list[int] | tuple[int, ...]) -> int:
    """整数列の積を返す。空なら 1（単位元）。"""
    result = 1
    for value in values:
        result *= int(value)
    return result


def validate_tt_matrix_cores(
    cores: list[torch.Tensor] | tuple[torch.Tensor, ...],
    out_modes: list[int] | tuple[int, ...],
    in_modes: list[int] | tuple[int, ...],
) -> None:
    """TT-matrix core 列の構造 contract を検証する。

    各 core:
        G^{(k)}.shape = (r_{k-1}, m_k, n_k, r_k)
        m_k = out_modes[k], n_k = in_modes[k]
        r_0 = r_d = 1

    引数:
        cores: 4階 TT-matrix cores（float64 / CPU）
        out_modes: 行側 physical modes (m_1, ..., m_d)
        in_modes: 列側 physical modes (n_1, ..., n_d)

    戻り値:
        None（不正なら TypeError / ValueError）
    """
    if not isinstance(cores, (list, tuple)):
        raise TypeError("cores must be a list or tuple.")

    d = len(cores)

    if d < 1:
        raise ValueError("TT-matrix must contain at least one core.")

    # site 数と mode 列の長さが一致すること
    if not (
        len(out_modes) == d
        and len(in_modes) == d
    ):
        raise ValueError(
            "len(cores), len(out_modes), and len(in_modes) must match."
        )

    for k, core in enumerate(cores):
        if not isinstance(core, torch.Tensor):
            raise TypeError(f"cores[{k}] must be a torch.Tensor.")

        # (r_left, m_k, n_k, r_right)
        if core.ndim != 4:
            raise ValueError(
                f"cores[{k}] must be 4D, but got shape {tuple(core.shape)}."
            )

        if core.dtype != torch.float64:
            raise ValueError(
                f"cores[{k}] must use torch.float64."
            )

        if core.device.type != "cpu":
            raise ValueError(
                f"cores[{k}] must be on CPU."
            )

        if core.shape[1] != out_modes[k]:
            raise ValueError(
                f"cores[{k}].shape[1] must equal out_modes[{k}]."
            )

        if core.shape[2] != in_modes[k]:
            raise ValueError(
                f"cores[{k}].shape[2] must equal in_modes[{k}]."
            )

        # 隣接 core の右 bond / 左 bond が一致
        if k > 0 and cores[k - 1].shape[3] != core.shape[0]:
            raise ValueError(
                f"Bond mismatch between cores[{k - 1}] and cores[{k}]."
            )

    # 境界 rank
    if cores[0].shape[0] != 1:
        raise ValueError("The first core must have left rank 1.")

    if cores[-1].shape[3] != 1:
        raise ValueError("The last core must have right rank 1.")


def tt_matrix_ranks(
    cores: list[torch.Tensor] | tuple[torch.Tensor, ...],
) -> list[int]:
    """TT-matrix の bond ranks を返す。

    戻り値:
        [r_0, r_1, ..., r_d]
        （通常 r_0 = r_d = 1）
    """
    if len(cores) < 1:
        raise ValueError("cores must not be empty.")

    return [
        int(cores[0].shape[0]),                 # r_0
        *[int(core.shape[3]) for core in cores],  # r_1, ..., r_d
    ]


def tt_matrix_num_parameters(
    cores: list[torch.Tensor] | tuple[torch.Tensor, ...],
) -> int:
    """TT-matrix cores の総要素数 Σ_k r_{k-1} m_k n_k r_k を返す。"""
    return int(sum(core.numel() for core in cores))


assert product([]) == 1
assert product([2, 3, 4]) == 24


## 15. $d=2$ の index 対応をコードで確認する

ここでは、

$$
m_1=2,\quad m_2=3,
$$

$$
n_1=3,\quad n_2=4
$$

とします。

したがって、

$$
m=6,\qquad n=12.
$$

`divmod` を使うと、

$$
i
\longleftrightarrow
(i_1,i_2)
$$

を直接確認できます。


In [48]:
m1, m2 = 2, 3
n1, n2 = 3, 4

print("row index mapping")
for i in range(m1 * m2):
    i1, i2 = divmod(i, m2)
    print(f"i={i:2d} -> (i1, i2)=({i1}, {i2})")

print()
print("column index mapping (first 8 only)")
for j in range(min(n1 * n2, 8)):
    j1, j2 = divmod(j, n2)
    print(f"j={j:2d} -> (j1, j2)=({j1}, {j2})")


row index mapping
i= 0 -> (i1, i2)=(0, 0)
i= 1 -> (i1, i2)=(0, 1)
i= 2 -> (i1, i2)=(0, 2)
i= 3 -> (i1, i2)=(1, 0)
i= 4 -> (i1, i2)=(1, 1)
i= 5 -> (i1, i2)=(1, 2)

column index mapping (first 8 only)
j= 0 -> (j1, j2)=(0, 0)
j= 1 -> (j1, j2)=(0, 1)
j= 2 -> (j1, j2)=(0, 2)
j= 3 -> (j1, j2)=(0, 3)
j= 4 -> (j1, j2)=(1, 0)
j= 5 -> (j1, j2)=(1, 1)
j= 6 -> (j1, j2)=(1, 2)
j= 7 -> (j1, j2)=(1, 3)


## 16. 検証用の $d=3$ TT-matrix cores

今回の主な数値例は、

$$
\text{out\_modes}=(2,3,2),
$$

$$
\text{in\_modes}=(2,2,3),
$$

$$
\text{TT-ranks}=(1,2,3,1)
$$

です。

core shapes は、

$$
G^{(1)}:(1,2,2,2),
$$

$$
G^{(2)}:(2,3,2,3),
$$

$$
G^{(3)}:(3,2,3,1)
$$

です。


In [49]:
out_modes = [2, 3, 2]
in_modes = [2, 2, 3]

cores = [
    torch.randn(
        1, 2, 2, 2,
        dtype=dtype,
        device=device,
    ),
    torch.randn(
        2, 3, 2, 3,
        dtype=dtype,
        device=device,
    ),
    torch.randn(
        3, 2, 3, 1,
        dtype=dtype,
        device=device,
    ),
]

validate_tt_matrix_cores(
    cores,
    out_modes,
    in_modes,
)

print("core shapes:")
for k, core in enumerate(cores, start=1):
    print(f"G{k}: {tuple(core.shape)}")

print("TT ranks:", tt_matrix_ranks(cores))
print("TT parameters:", tt_matrix_num_parameters(cores))
print("dense shape:", (product(out_modes), product(in_modes)))
print("dense parameters:", product(out_modes) * product(in_modes))


core shapes:
G1: (1, 2, 2, 2)
G2: (2, 3, 2, 3)
G3: (3, 2, 3, 1)
TT ranks: [1, 2, 3, 1]
TT parameters: 62
dense shape: (12, 12)
dense parameters: 144


## 17. 演習1 — `tt_matrix_to_dense`

### 目的

TT-matrix cores が表す dense weight、

$$
W\in\mathbb R^{m\times n}
$$

を小規模な検証用に復元します。

### 重要

この関数は **検証専用**です。

`tt_linear_forward` の中から呼んではいけません。

### 中間 axis 順序

第 $k$ site まで contract した intermediate は、

$$
(i_1,j_1,\ldots,i_k,j_k,\alpha_k)
$$

の順を保ちます。

第 $k+1$ core、

$$
(\alpha_k,i_{k+1},j_{k+1},\alpha_{k+1})
$$

と bond $\alpha_k$ を縮約すると、

$$
(i_1,j_1,\ldots,i_{k+1},j_{k+1},\alpha_{k+1})
$$

になります。

最後は、

$$
(i_1,j_1,\ldots,i_d,j_d)
$$

です。

その後、

$$
(i_1,j_1,\ldots,i_d,j_d)
\rightarrow
(i_1,\ldots,i_d,j_1,\ldots,j_d)
$$

へ `permute` し、

$$
(m,n)
$$

へ reshape します。

### TODO

`tt_matrix_to_dense(cores, out_modes, in_modes, verbose=False)` を実装してください。

### 要件

- 入力 core を変更しない
- `validate_tt_matrix_cores` を使う
- `verbose=True` で各 intermediate shape を表示
- 最終 permutation をプログラム的に作る
- 出力 shape が `(m, n)` であることを assert


In [50]:
def interleaved_tt_matrix_tensor_to_dense(
    interleaved: torch.Tensor,
    out_modes: list[int],
    in_modes: list[int],
) -> torch.Tensor:
    """interleaved TT-matrix tensor を dense weight W へ変換する。

    interleaved:
        shape = (m1, n1, m2, n2, ..., md, nd)
        axes  = (i1, j1, i2, j2, ..., id, jd)

    returns:
        W_dense:
            shape = (m, n)
            m = product(out_modes)
            n = product(in_modes)
    """
    d = len(out_modes)

    # --- 入力チェック ---
    assert len(in_modes) == d
    assert interleaved.ndim == 2 * d  # site ごとに (m_k, n_k) の2軸

    # 期待 shape: (m1, n1, m2, n2, ..., md, nd)
    expected_interleaved_shape = tuple(
        size
        for mk, nk in zip(out_modes, in_modes)
        for size in (mk, nk)
    )
    assert interleaved.shape == expected_interleaved_shape, (
        f"interleaved.shape={tuple(interleaved.shape)}, "
        f"expected={expected_interleaved_shape}"
    )

    # --- interleaved → grouped の軸並べ替え ---
    # (i1, j1, i2, j2, ..., id, jd)
    # → (i1, i2, ..., id, j1, j2, ..., jd)
    # 偶数番軸 = 行 index i_k、奇数番軸 = 列 index j_k
    perm_interleaved_to_grouped = (
        list(range(0, 2 * d, 2))  # i1, i2, ..., id
        + list(range(1, 2 * d, 2))  # j1, j2, ..., jd
    )

    grouped = interleaved.permute(*perm_interleaved_to_grouped)

    # --- dense 行列 (m, n) へ畳む ---
    m = product(out_modes)  # 行数 = Π m_k
    n = product(in_modes)   # 列数 = Π n_k

    W_dense = grouped.reshape(m, n)

    assert W_dense.shape == (m, n)

    return W_dense

In [51]:
# TODO 1:
# tt_matrix_to_dense を実装してください。
#

def clone_cores(cores):
    return [G.clone() for G in cores]

def tt_matrix_to_dense(
    cores: list[torch.Tensor] | tuple[torch.Tensor, ...],
    out_modes: list[int] | tuple[int, ...],
    in_modes: list[int] | tuple[int, ...],
    *,
    verbose: bool = False,
) -> torch.Tensor:
    """TT-matrix cores から dense weight W を復元する（検証専用）。

    処理の流れ:
        1. validate
        2. 左→右に bond 縮約（interleaved: i1,j1,...,ik,jk,α）
        3. 最終右 rank 1 を落とす
        4. interleaved → grouped へ permute
        5. (m, n) へ reshape

    引数:
        cores: TT-matrix core 列
            G^{(k)}.shape = (r_{k-1}, m_k, n_k, r_k),
            r_0 = r_d = 1
        out_modes: 行側 physical modes (m_1, ..., m_d)
            m = product(out_modes)
        in_modes: 列側 physical modes (n_1, ..., n_d)
            n = product(in_modes)
        verbose: True なら各 intermediate shape を表示

    戻り値:
        W: dense weight, shape (m, n), dtype float64, device CPU

    注意:
        本番の tt_linear_forward からは呼ばないこと。
    """
    cores = clone_cores(cores)
    validate_tt_matrix_cores(cores, out_modes, in_modes)

    # 左→右 contraction（interleaved を保つ）
    current = cores[0].squeeze(0)  # (1,m1,n1,r1) → (m1,n1,r1)
    if verbose:
        print("after core 0:", tuple(current.shape))

    for k, core in enumerate(cores[1:], start=1):
        current = torch.tensordot(current, core, dims=([-1], [0]))
        if verbose:
            print(f"after core {k}:", tuple(current.shape))

    # TODO: 最終右 rank 1 を落とす
    current = current.squeeze(-1) 
    if verbose:
        print("interleaved:", tuple(current.shape))
    return interleaved_tt_matrix_tensor_to_dense(current, list(out_modes), list(in_modes))


## 18. 演習2 — Dense Reconstruction の shape 遷移

### 目的

$d=3$ の cores で `verbose=True` を使い、

$$
(i_1,j_1,\alpha_1)
$$

$$
\rightarrow
(i_1,j_1,i_2,j_2,\alpha_2)
$$

$$
\rightarrow
(i_1,j_1,i_2,j_2,i_3,j_3)
$$

という shape 遷移を確認します。

### 期待する intermediate shape

最初の core の left rank を除くと、

$$
(2,2,2)
$$

です。

次の core を contract すると、

$$
(2,2,3,2,3)
$$

です。

最後の core を contract すると、

$$
(2,2,3,2,2,3,1)
$$

となり、最後の right rank 1 を除いて、

$$
(2,2,3,2,2,3)
$$

という interleaved tensor になります。

### TODO

- `tt_matrix_to_dense(..., verbose=True)` を実行
- dense shape `(12, 12)` を確認


In [52]:
# TODO 2:
# verbose=True で tt_matrix_to_dense を実行し、
# shape 遷移と最終 shape を確認してください。
#
# 期待する intermediate（verbose 出力）:
#   after core 0: (2, 2, 2)
#   after core 1: (2, 2, 3, 2, 3)
#   after core 2: (2, 2, 3, 2, 2, 3, 1)  ※squeeze 前ならこれ
#   interleaved:  (2, 2, 3, 2, 2, 3)
# 最終 dense: (12, 12)
#
W = tt_matrix_to_dense(cores, out_modes, in_modes, verbose=True)

print("dense W.shape:", tuple(W.shape))
assert W.shape == (12, 12)
assert W.shape == (product(out_modes), product(in_modes))
print("OK: shape transition + final (12, 12)")


after core 0: (2, 2, 2)
after core 1: (2, 2, 3, 2, 3)
after core 2: (2, 2, 3, 2, 2, 3, 1)
interleaved: (2, 2, 3, 2, 2, 3)
dense W.shape: (12, 12)
OK: shape transition + final (12, 12)


## 19. 演習3 — Dense Weight の1要素を core contraction と照合する

### 目的

復元した dense $W$ の特定要素が、

$$
\sum_{\alpha_1,\alpha_2}
G^{(1)}_{0,i_1,j_1,\alpha_1}
G^{(2)}_{\alpha_1,i_2,j_2,\alpha_2}
G^{(3)}_{\alpha_2,i_3,j_3,0}
$$

と一致することを確認します。

### multi-index

任意に、

$$
(i_1,i_2,i_3)
$$

と、

$$
(j_1,j_2,j_3)
$$

を選びます。

flat index は、

$$
i=i_1(m_2m_3)+i_2m_3+i_3,
$$

$$
j=j_1(n_2n_3)+j_2n_3+j_3.
$$

### TODO

- dense 側 `W[i, j]`
- core contraction 側
- `torch.testing.assert_close`

を比較してください。


In [53]:
# TODO 3:
# dense W の1要素と、
# 3 core の明示的 bond contraction を比較してください。
#
# --- 設定 ---
# multi-index（任意に選ぶ）
i1, i2, i3 = 1, 0, 1
j1, j2, j3 = 0, 1, 2

m1, m2, m3 = out_modes
n1, n2, n3 = in_modes

# flat index
# i = i1*(m2*m3) + i2*m3 + i3
# j = j1*(n2*n3) + j2*n3 + j3
i = i1 * (m2 * m3) + i2 * m3 + i3
j = j1 * (n2 * n3) + j2 * n3 + j3

W = tt_matrix_to_dense(cores, out_modes, in_modes)
G1, G2, G3 = cores

print("multi-index i:", (i1, i2, i3), "-> flat i =", i)
print("multi-index j:", (j1, j2, j3), "-> flat j =", j)
print("W[i, j] (dense):", W[i, j].item())

# --- ここから先（自分で） ---
# 1. core contraction:
A1 = G1[0, i1, j1, :]
A2 = G2[:, i2, j2, :]
A3 = G3[:, i3, j3, 0]
print(f"A1.shape: {tuple(A1.shape)}, A2.shape: {tuple(A2.shape)}, A3.shape: {tuple(A3.shape)}")
val_cores = torch.einsum("a,ab,b->", A1, A2, A3)
# 2. torch.testing.assert_close(W[i, j], val_cores)
torch.testing.assert_close(
    W[i, j],
    val_cores,
    rtol=1e-12,
    atol=1e-12,
)
print("PASS: dense W[i, j] and explicit TT-core contraction agree.")


multi-index i: (1, 0, 1) -> flat i = 7
multi-index j: (0, 1, 2) -> flat j = 5
W[i, j] (dense): -0.5731065395929638
A1.shape: (2,), A2.shape: (2, 3), A3.shape: (3,)
PASS: dense W[i, j] and explicit TT-core contraction agree.


## 20. 演習4 — Interleaved / Grouped permutation の検証

$d=3$ では interleaved axis は、

$$
(i_1,j_1,i_2,j_2,i_3,j_3)
$$

です。

grouped axis は、

$$
(i_1,i_2,i_3,j_1,j_2,j_3)
$$

です。

したがって permutation は、

$$
\boxed{
[0,2,4,1,3,5]
}
$$

です。

### TODO

1. interleaved tensor の shape を表示
2. grouped tensor の shape を表示
3. 任意の multi-index について、

$$
\text{interleaved}[i_1,j_1,i_2,j_2,i_3,j_3]
$$

と、

$$
\text{grouped}[i_1,i_2,i_3,j_1,j_2,j_3]
$$

が一致することを `assert_close` で確認してください。


In [54]:
# TODO 4:
# interleaved -> grouped permutation を明示的に検証してください。
#
# --- 設定 ---
# d=3 の期待 perm: [0, 2, 4, 1, 3, 5]
#   interleaved (i1,j1,i2,j2,i3,j3)
#   → grouped   (i1,i2,i3,j1,j2,j3)
#
d = len(out_modes)
assert d == 3

# cores から interleaved を作る（右 rank を落としたところまで）
_cores = [G.clone() for G in cores]
current = _cores[0].squeeze(0)
for core in _cores[1:]:
    current = torch.tensordot(current, core, dims=([-1], [0]))
interleaved = current.squeeze(-1)  # (m1,n1,m2,n2,m3,n3)

perm = [0, 2, 4, 1, 3, 5]
grouped = interleaved.permute(*perm)

# 検証用 multi-index（TODO 3 と同じでも可）
i1, i2, i3 = 1, 0, 1
j1, j2, j3 = 0, 1, 2

print("interleaved.shape:", tuple(interleaved.shape))
print("grouped.shape:", tuple(grouped.shape))
print("perm:", perm)
print("interleaved[i1,j1,i2,j2,i3,j3]:", interleaved[i1, j1, i2, j2, i3, j3].item())
print("grouped[i1,i2,i3,j1,j2,j3]:", grouped[i1, i2, i3, j1, j2, j3].item())

torch.testing.assert_close(
    interleaved[i1, j1, i2, j2, i3, j3],
    grouped[i1, i2, i3, j1, j2, j3],
)


interleaved.shape: (2, 2, 3, 2, 2, 3)
grouped.shape: (2, 3, 2, 2, 2, 3)
perm: [0, 2, 4, 1, 3, 5]
interleaved[i1,j1,i2,j2,i3,j3]: -0.5731065395929638
grouped[i1,i2,i3,j1,j2,j3]: -0.5731065395929638


## 21. 演習5 — `tt_linear_forward`

### 目的

dense weight を復元せず、

$$
Y=XW^{\mathsf T}+b
$$

を TT-matrix core contraction だけで計算します。

### API

```python
tt_linear_forward(
    x,
    cores,
    out_modes,
    in_modes,
    bias=None,
    verbose=False,
)
```

### 入力

$$
x\in\mathbb R^{B\times n}
$$

です。

最初に、

$$
(B,n_1,\ldots,n_d)
$$

へ reshape します。

### 収縮方向

今回は **右から左**へ処理します。

$d=3$ では、

$$
(B,n_1,n_2,n_3)
$$

$$
\rightarrow
(B,n_1,n_2,m_3,r_2)
$$

$$
\rightarrow
(B,n_1,m_2,m_3,r_1)
$$

$$
\rightarrow
(B,m_1,m_2,m_3)
$$

です。

### TODO

`tt_linear_forward` を一般の $d$ で実装してください。

### 要件

- `tt_matrix_to_dense` を呼ばない
- dense $W$ を生成しない
- x は2次元 / CPU / float64
- `x.shape[1] == product(in_modes)`
- bias は `None` または shape `(m,)`
- core は右から左へ処理
- `verbose=True` で shape と axis の意味を表示
- 最終 shape `(B, m)`
- 入力 core を変更しない
- `torch.einsum` の文字列はプログラム的に生成
- 学習用として `d <= 8` としてよい


In [55]:
# TODO 5:
# dense reconstruction を使わない tt_linear_forward を実装してください。
#
def tt_linear_forward(
    x: torch.Tensor,
    cores: list[torch.Tensor] | tuple[torch.Tensor, ...],
    out_modes: list[int] | tuple[int, ...],
    in_modes: list[int] | tuple[int, ...],
    bias: torch.Tensor | None = None,
    *,
    verbose: bool = False,
) -> torch.Tensor:
    """TT-matrix cores だけで Y = X W^T + b を計算する（dense W は作らない）。

    処理の流れ（右→左 contraction）:
        1. validate cores / x / bias
        2. x: (B, n) → (B, n_1, ..., n_d)
        3. site d, d-1, ..., 1 の順に einsum
        4. (B, m_1, ..., m_d) → (B, m)
        5. bias があれば加算

    引数:
        x: 入力, shape (B, n), n = product(in_modes)
            dtype float64, device CPU
        cores: TT-matrix core 列
            G^{(k)}.shape = (r_{k-1}, m_k, n_k, r_k)
        out_modes: 行側 modes (m_1, ..., m_d), m = product(out_modes)
        in_modes: 列側 modes (n_1, ..., n_d)
        bias: None または shape (m,)
        verbose: True なら各 intermediate shape を表示

    戻り値:
        y: shape (B, m)

    注意:
        tt_matrix_to_dense を呼んではいけない。
        einsum 文字列はプログラム的に生成する（学習用 d <= 8）。
    """
    # TODO: 実装
    #TT-matrix core 列の構造 contract を検証する
    validate_tt_matrix_cores(cores,out_modes,in_modes)

    # 2. 入力は２次元か
    assert 2 == x.dim(), (
        f"入力は２次元でない x.dim={x.dim}"
    )
    # XがCPUでなくてもいいので、条件には加えない。
    assert x.dtype == torch.float64, (
        f"dtype が一致しません: x.dtype={x.dtype}"
    )
    assert x.shape[1] == product(in_modes), (
        f"in_modesがXのin_modeと一致しません: x.shape[1]={x.shape[1]}, product(in_modes)={product(in_modes)}"
    )
    assert bias.shape == (product(out_modes),),(
        f"biasがXのout_modeと一致しません: bias.shape={bias.shape}, product(in_modes)={product(out_modes)}"
    )

    batch_size = x.shape[0]
    state = x.reshape(batch_size, *in_modes)
    assert state.shape == (batch_size, *in_modes), (
        f"state.shape={tuple(state.shape)}, "
        f"expected={batch_size, *in_modes}"
    )
    # last core
    # state: (B, n_1, ..., n_d) / core: (r_{d-1}, m_d, n_d, 1)
    # g(=d軸の右 bond) は 1
    next_state = torch.einsum(
        "...a,bcad->...cb",
        state,
        cores[-1],
    )

    assert next_state.shape == (
        batch_size,
        *in_modes[:-1],
        out_modes[-1],
        cores[-1].shape[0],  # r_{d-1}
    )
    # center core
    for k in range(len(cores)-2,0,-1):
        next_state = next_state.movedim(
            k+1,
            next_state.ndim - 2,
        )
        next_state = torch.einsum(
            "...ja,pija->...ip",
            next_state,
            cores[k],
        )
    # first core
    next_state = next_state.movedim(1, next_state.ndim - 2)  # n1
    # pは最初のボンド(1)
    next_state = torch.einsum(
        "...ja,pija->...i",
        next_state,
        cores[0],
    )

    # 軸0（B）以外を逆順に並べて permute
    # (B, m_d, ..., m_1) → (B, m_1, ..., m_d) → (B, m)
    d = len(cores)
    perm = [0] + list(range(d, 0, -1))
    next_state = next_state.permute(*perm)
    # (B, m1, ..., md) を (B, m) に畳む。m = product(out_modes)
    y = next_state.reshape(batch_size, product(out_modes))
    if bias is not None:
        y = y + bias
    return y


## 22. 演習6 — TT-Linear Forward の shape 遷移

batch size を、

$$
B=4
$$

とします。

入力 feature 数は、

$$
n=2\cdot2\cdot3=12
$$

なので、

$$
x\in\mathbb R^{4\times12}
$$

です。

### 期待 shape

入力 reshape：

$$
(4,2,2,3)
$$

core 3 後：

$$
(4,2,2,2,3)
$$

core 2 後：

$$
(4,2,3,2,2)
$$

core 1 後：

$$
(4,2,3,2)
$$

最後に、

$$
(4,12)
$$

へ flatten します。

### TODO

- ランダム `x`
- ランダム `bias`
- `verbose=True`
- shape の print / assert

を行ってください。


In [56]:
# TODO 6:
# batch size 4 の入力で tt_linear_forward の shape 遷移を確認してください。
#
# --- 設定 ---
# 期待 shape（理論）:
#   input reshape: (4, 2, 2, 3)
#   after core 3:  (4, 2, 2, 2, 3)   # (B, n1, n2, m3, r2)
#   after core 2:  (4, 2, 3, 2, 2)   # (B, n1, m2, m3, r1)
#   after core 1:  (4, 2, 3, 2)     # (B, m1, m2, m3)
#   final y:       (4, 12)
#
B = 4
n = product(in_modes)   # 2*2*3 = 12
m = product(out_modes)  # 2*3*2 = 12

x = torch.randn(B, n, dtype=dtype, device=device)
bias = torch.randn(m, dtype=dtype, device=device)

print("x.shape:", tuple(x.shape))
print("bias.shape:", tuple(bias.shape))

y = tt_linear_forward(x, cores, out_modes, in_modes, bias=bias, verbose=True)
assert y.shape == (B, m)


x.shape: (4, 12)
bias.shape: (12,)


## 23. 演習7 — Dense Linear / `F.linear` との数値一致

### 目的

同じ TT cores が定義する $W_{\mathrm{TT}}$ について、

1. TT forward
2. dense 行列積
3. `F.linear`

を比較します。

### 計算

$$
Y_{\mathrm{TT}}
=
\operatorname{tt\_linear\_forward}(...)
$$

$$
W_{\mathrm{TT}}
=
\operatorname{tt\_matrix\_to\_dense}(...)
$$

$$
Y_{\mathrm{dense}}
=
XW_{\mathrm{TT}}^{\mathsf T}+b
$$

$$
Y_{\mathrm{F.linear}}
=
\operatorname{F.linear}(X,W_{\mathrm{TT}},b)
$$

### TODO

次を必ず確認してください。

```python
torch.testing.assert_close(
    y_tt,
    y_dense,
    rtol=1e-12,
    atol=1e-12,
)

torch.testing.assert_close(
    y_tt,
    y_f_linear,
    rtol=1e-12,
    atol=1e-12,
)

torch.testing.assert_close(
    y_dense,
    y_f_linear,
    rtol=1e-12,
    atol=1e-12,
)
```

最大絶対誤差も表示してください。

### 注意

比較対象は「元の学習済み dense weight」ではなく、

> **今回の random TT-matrix cores が定義する dense $W_{\mathrm{TT}}$**

です。


In [57]:
# TODO 7:
# y_tt / y_dense / y_f_linear を比較してください。
#
# --- 設定 ---
# 直前セルの x / bias / cores / out_modes / in_modes を使う
import torch.nn.functional as F

y_tt = tt_linear_forward(x, cores, out_modes, in_modes, bias=bias, verbose=True)
W_tt = tt_matrix_to_dense(cores, out_modes, in_modes)
y_dense = torch.tensordot(x, W_tt.T, dims=([-1], [0])) + bias
y_f_linear = F.linear(x, W_tt, bias)

print("|y_tt - y_dense|_max:", (y_tt - y_dense).abs().max().item())
print("|y_tt - y_f_linear|_max:", (y_tt - y_f_linear).abs().max().item())
print("|y_dense - y_f_linear|_max:", (y_dense - y_f_linear).abs().max().item())

torch.testing.assert_close(y_tt, y_dense, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(y_tt, y_f_linear, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(y_dense, y_f_linear, rtol=1e-12, atol=1e-12)
print("OK: y_tt ≈ y_dense ≈ y_f_linear")


|y_tt - y_dense|_max: 5.329070518200751e-15
|y_tt - y_f_linear|_max: 5.329070518200751e-15
|y_dense - y_f_linear|_max: 3.552713678800501e-15
OK: y_tt ≈ y_dense ≈ y_f_linear


## 24. 演習8 — Rank 1 / $d=2$ と Kronecker 積

理解補助として、

$$
d=2,
\qquad
r_1=1
$$

を考えます。

$$
A\in\mathbb R^{m_1\times n_1},
$$

$$
B\in\mathbb R^{m_2\times n_2}
$$

から、

$$
G^{(1)}
=
A.\operatorname{reshape}(1,m_1,n_1,1),
$$

$$
G^{(2)}
=
B.\operatorname{reshape}(1,m_2,n_2,1)
$$

を作ります。

このとき、

$$
\boxed{
W_{\mathrm{TT}}
=
A\otimes B
}
$$

です。

### TODO

設定：

$$
m_1=m_2=n_1=n_2=2.
$$

確認：

```python
torch.testing.assert_close(
    W_tt,
    torch.kron(A, B),
    rtol=1e-12,
    atol=1e-12,
)
```

さらにランダム `x` / `bias` について、

$$
Y_{\mathrm{TT}}
$$

と、

$$
F.linear(x,\operatorname{kron}(A,B),b)
$$

を比較してください。


In [58]:
# TODO 8:
# d=2, rank=1 の Kronecker 積ケースを検証してください。
#
# --- 設定 ---
m1 = m2 = n1 = n2 = 2
out_modes_k = [m1, m2]
in_modes_k = [n1, n2]

A = torch.randn(m1, n1, dtype=dtype, device=device)
B = torch.randn(m2, n2, dtype=dtype, device=device)

# 1. rank-1 TT-matrix cores
cores_k = [
    A.reshape(1, m1, n1, 1),
    B.reshape(1, m2, n2, 1),
]
validate_tt_matrix_cores(cores_k, out_modes_k, in_modes_k)

# 2. TT → dense W
W_tt = tt_matrix_to_dense(cores_k, out_modes_k, in_modes_k)
print("W_tt.shape:", tuple(W_tt.shape))

# 3. Kronecker と比較
W_kron = torch.kron(A, B)
print("W_kron.shape:", tuple(W_kron.shape))
torch.testing.assert_close(W_tt, W_kron, rtol=1e-12, atol=1e-12)
print("OK: W_tt ≈ kron(A, B)")

# 4. forward 比較
import torch.nn.functional as F

x_k = torch.randn(4, product(in_modes_k), dtype=dtype, device=device)
bias_k = torch.randn(product(out_modes_k), dtype=dtype, device=device)

y_tt_k = tt_linear_forward(x_k, cores_k, out_modes_k, in_modes_k, bias=bias_k)
y_f_k = F.linear(x_k, W_kron, bias_k)

print("|y_tt - F.linear|_max:", (y_tt_k - y_f_k).abs().max().item())
torch.testing.assert_close(y_tt_k, y_f_k, rtol=1e-12, atol=1e-12)
print("OK: tt_linear_forward ≈ F.linear(x, kron(A,B), bias)")


W_tt.shape: (4, 4)
W_kron.shape: (4, 4)
OK: W_tt ≈ kron(A, B)
|y_tt - F.linear|_max: 8.881784197001252e-16
OK: tt_linear_forward ≈ F.linear(x, kron(A,B), bias)


## 25. Dense Weight を TT-matrix 用 Tensor へ変換する

既存 dense weight、

$$
W\in\mathbb R^{m\times n}
$$

を TT-SVD へ渡すには、まず通常 TT tensor へ変換する必要があります。

### Step 1：Grouped reshape

$$
W
\rightarrow
\mathcal W_{\text{grouped}}
$$

shape：

$$
(m_1,\ldots,m_d,n_1,\ldots,n_d)
$$

axis：

$$
(i_1,\ldots,i_d,j_1,\ldots,j_d)
$$

### Step 2：Interleave

$$
(i_1,\ldots,i_d,j_1,\ldots,j_d)
$$

$$
\longrightarrow
$$

$$
(i_1,j_1,\ldots,i_d,j_d)
$$

### Step 3：Siteごとに combine

$$
q_k=m_kn_k
$$

として、

$$
\boxed{
\mathcal A
\in
\mathbb R^{q_1\times\cdots\times q_d}
}
$$

へ reshape します。

この $\mathcal A$ が既存 TT-SVD の入力になります。


## 26. 演習9 — `dense_to_tt_matrix_tensor`

### API

```python
dense_to_tt_matrix_tensor(
    W,
    out_modes,
    in_modes,
    verbose=False,
)
```

### 処理

1. $W$ の shape `(m, n)` を検証
2. grouped shape へ reshape
3. grouped → interleaved に permute
4. 各 `(m_k, n_k)` を combine
5. shape `(q_1, ..., q_d)` を返す

### TODO

`dense_to_tt_matrix_tensor` を実装してください。

### 要件

- input `W` を変更しない
- CPU / float64
- `W.shape == (product(out_modes), product(in_modes))`
- `verbose=True` で
  - grouped shape
  - permutation
  - interleaved shape
  - combined shape

  を表示
- TT-SVD 自体はここでは実装しない


In [59]:
# TODO 9:
# dense_to_tt_matrix_tensor を実装してください。
#
def dense_to_tt_matrix_tensor(
    W: torch.Tensor,
    out_modes: list[int] | tuple[int, ...],
    in_modes: list[int] | tuple[int, ...],
    *,
    verbose: bool = False,
) -> torch.Tensor:
    """dense weight W を TT-SVD 入力用の combined tensor へ並べ替える。

    処理の流れ:
        1. W.shape == (m, n) を検証（m,n = product(out/in_modes)）
        2. grouped: (m_1,...,m_d, n_1,...,n_d) へ reshape
        3. grouped → interleaved: (m_1,n_1,...,m_d,n_d) へ permute
        4. 各 site で (m_k, n_k) → q_k = m_k * n_k に combine
        5. shape (q_1, ..., q_d) を返す

    引数:
        W: dense weight, shape (m, n)
            dtype float64, device CPU（入力は変更しない）
        out_modes: 行側 modes (m_1, ..., m_d)
        in_modes: 列側 modes (n_1, ..., n_d)
        verbose: True なら grouped / perm / interleaved / combined を表示

    戻り値:
        A: combined tensor, shape (q_1, ..., q_d)
            q_k = m_k * n_k

    注意:
        TT-SVD 自体はこの関数では行わない（並べ替えのみ）。
    """
    # TODO: 実装
    raise NotImplementedError


## 27. 演習10 — Tensorization の要素対応を確認する

### 目的

`dense_to_tt_matrix_tensor` が単に shape を変えているだけではなく、**正しい site pairing** を作っていることを確認します。

$d=3$ なら combined tensor の index、

$$
(a_1,a_2,a_3)
$$

は、

$$
a_k=i_kn_k+j_k
$$

に対応します。

### TODO

任意の、

$$
(i_1,i_2,i_3,j_1,j_2,j_3)
$$

を選び、

1. 元の dense $W[i,j]$
2. combined tensor $A[a_1,a_2,a_3]$

が一致することを確認してください。


In [60]:
# TODO 10:
# dense W と combined TT-SVD input tensor の要素対応を確認してください。


## 28. TT-SVD 初期化への接続

ここまでで、

$$
W_{\mathrm{original}}
\in
\mathbb R^{m\times n}
$$

から、

$$
\mathcal A
\in
\mathbb R^{q_1\times\cdots\times q_d},
\qquad
q_k=m_kn_k
$$

を作れるようになります。

次は既存 TT-SVD を、

$$
\mathcal A
$$

へ適用します。

得られる通常 TT core は、

$$
C^{(k)}
\in
\mathbb R^{r_{k-1}\times q_k\times r_k}
$$

です。

これを、

$$
\boxed{
G^{(k)}
=
C^{(k)}
.\operatorname{reshape}
(r_{k-1},m_k,n_k,r_k)
}
$$

とすれば TT-matrix core になります。

### Full-rank の場合

full-rank TT-SVD なら、

$$
W_{\mathrm{original}}
\approx
W_{\mathrm{TT}}
$$

で、差は float64 の丸め誤差程度になるはずです。

### Truncated の場合

rank を切り詰めた場合は、

$$
\boxed{
\frac{
\|W_{\mathrm{original}}-W_{\mathrm{TT}}\|_F
}{
\|W_{\mathrm{original}}\|_F
}
}
$$

を評価します。

今回は TT-SVD 自体の再実装は行いません。


## 29. 入力 validation で重要なもの

今回の関数では、次を明示的に確認します。

### TT-matrix cores

- list / tuple
- 1個以上
- 各 core は4階
- axis 1 が `out_modes[k]`
- axis 2 が `in_modes[k]`
- 隣接 bond が一致
- 最初の left rank が1
- 最後の right rank が1
- CPU
- float64

### `tt_linear_forward`

- `x.ndim == 2`
- CPU / float64
- `x.shape[1] == product(in_modes)`
- bias は `None` または shape `(m,)`
- bias も CPU / float64

### `dense_to_tt_matrix_tensor`

- $W$ は2次元
- shape `(m, n)`
- CPU / float64

TT-rank の値そのものは、隣接 core の bond が整合していれば任意です。


## 30. 今回のまとめ

### TT-matrix core

$$
\boxed{
G^{(k)}
\in
\mathbb R^{r_{k-1}\times m_k\times n_k\times r_k}
}
$$

axis は、

$$
(\alpha_{k-1},i_k,j_k,\alpha_k)
$$

です。

### Dense Weight との対応

$$
i\leftrightarrow(i_1,\ldots,i_d),
$$

$$
j\leftrightarrow(j_1,\ldots,j_d).
$$

dense weight を TT-matrix 用に扱うには、

$$
(i_1,\ldots,i_d,j_1,\ldots,j_d)
$$

を、

$$
(i_1,j_1,\ldots,i_d,j_d)
$$

へ interleave します。

### TT-Linear Forward

$$
X\in\mathbb R^{B\times n}
$$

を、

$$
(B,n_1,\ldots,n_d)
$$

へ reshape し、各 input mode $j_k$ を TT core と縮約し、

$$
(i_1,\ldots,i_d)
$$

を生成します。

最終的に、

$$
\boxed{
Y=XW^{\mathsf T}+b
}
$$

と同じ結果を、dense $W$ を forward 中に作らずに求めます。

### 次

この Notebook が完了したら、

1. dense `nn.Linear.weight`
2. TT-matrix 用 tensorization
3. 既存 TT-SVD
4. TT-matrix core
5. TT-Linear forward

をつなぎ、実際の `nn.Linear` を TT 化する段階へ進めます。

その後、Fashion-MNIST の MLP に戻ります。
